# Lecture 5 — Expectations and Transition Dynamics
## 第五讲 —— 预期与过渡动态

**Computational Methods for Heterogeneous-Agent Macro**  
**异质性主体宏观的计算方法**

Jeffrey Sun

### Environment
### 运行环境

In [1]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using Plots, Printf
using LinearAlgebra
using HouseholdStages
using Accessors

  Activating project at `~/projects/research/CV is all you need`


## 1. What is `HouseholdStages`?
## 1. 什么是 `HouseholdStages`？

A Julia package containing optimized implementations of Stages.

一个包含 Stage 优化实现的 Julia 包。

- **Stage primitives** — `MarkovStage`, `WealthChangeStage`,
  `ConsumptionSavingsStage`, `LogitChoiceStage`, `MigrationStage`,
  `BorrowingConstraintStage`, …
- **Composition** — `∘ₛ` chains Stages in time order;
  `lift_moments` attaches integrals against the chain's terminal $\Lambda$.
- **Three inner-solve helpers** — `solve_vfi_steady_state_given_env!`,
  `solve_lambda_steady_state_given_env!`,
  `solve_steady_state_given_env!` (a bundle of the two).
- **GPU** and **automatic differentiation** (not today).

- **Stage 基元** —— `MarkovStage`、`WealthChangeStage`、
  `ConsumptionSavingsStage`、`LogitChoiceStage`、`MigrationStage`、
  `BorrowingConstraintStage`……
- **组合** —— `∘ₛ` 把 Stage 按时间顺序串起来；
  `lift_moments` 把要计算的矩附加到链末端的 $\Lambda$ 上。
- **三个内层求解辅助函数** —— `solve_vfi_steady_state_given_env!`、
  `solve_lambda_steady_state_given_env!`、
  `solve_steady_state_given_env!`（前两者的打包）。
- **GPU** 与**自动微分**（今天不讲）。

## 2. Aiyagari model in `HouseholdStages`
## 2. 用 `HouseholdStages` 复现 Aiyagari 稳态

Same model as L04: Three-Stage household problem, Cobb-Douglas firm, equilibrium $\bar K = \bar K^{\mathrm{supplied}}$.

与 L04 相同的模型：三阶段家庭问题、Cobb-Douglas 企业、均衡 $\bar K = \bar K^{\mathrm{supplied}}$。

### 2.1 Parameters and layout / 参数与状态空间

In [ ]:
"Data structure (struct) for the parameters of the Aiyagari model. / Aiyagari 模型参数的数据结构（struct）。"
@kwdef struct AiyagariParams
    β ::Float64 = 0.96
    σ ::Float64 = 1.5
    α ::Float64 = 0.36
    δ ::Float64 = 0.08
    L ::Float64 = 1.0
    A ::Float64 = 1.0
    y_grid ::Vector{Float64} = [0.6, 1.0, 1.4]
    P_y    ::Matrix{Float64} = [0.7 0.2 0.1;
                                 0.2 0.6 0.2;
                                 0.1 0.2 0.7]
    N_w   ::Int     = 400
    w_min ::Float64 = 0.0
    w_max ::Float64 = 100.0
end
#Base.Broadcast.broadcastable(p::AiyagariParams) = Ref(p)

"Array layout for the household state space in the Aiyagari model. / Aiyagari 模型中家庭状态空间的数组布局。"
function aiyagari_layout(p::AiyagariParams)
    return StateLayout(
        StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length = p.N_w, spacing = :log)),
        StateAxis(:income, p.y_grid),
    )
end

In [3]:
p = AiyagariParams()
dims = layout_size(aiyagari_layout(p))

(400, 3)

### 2.2 The three Stages / 三个 Stage

1. Markov income shock
2. Receive income
3. Consumption-savings choice

1. Markov 收入冲击
2. 领取收入
3. 消费—储蓄选择

In [ ]:
_u_crra(c, ::Val{1}) = log(c)
_u_crra(c, ::Val{σ}) where σ = (c^(1 - σ)) / (1 - σ)
u_crra(c, valσ) = c < 0 ? -Inf : _u_crra(c, valσ)

function aiyagari_household(p::AiyagariParams)
    layout = aiyagari_layout(p)

    # Stage 1 — Markov Income Shock
    # 第一阶段 —— Markov 收入冲击
    shock = MarkovStage(layout; axis = :income, transition = p.P_y)

    # Stage 2 — Receive Income
    # 第二阶段 —— 领取收入
    receipt = WealthChangeStage(layout;
        wealth_post  = (cell; env) -> (1 + env.r) * cell.wealth + env.w * cell.income,
        wealth_axis  = :wealth, # Not strictly necessary--set by default / 不是必需的——默认就是这个
    )

    # Stage 3 — Consumption Savings
    # 第三阶段 —— 消费储蓄
    savings = ConsumptionSavingsStage(layout;
        β               = p.β,
        utility         = (cell, c; env) -> u_crra(c, Val(p.σ)),
        wealth_axis     = :wealth, # Not strictly necessary--set by default / 不是必需的——默认就是这个
        monotone_search = :divide_conquer,
    )

    # Construct household block by composition
    # 用 Stage 复合构造家庭模块
    hh = shock ∘ₛ receipt ∘ₛ savings

    # Register a moment to be computed--in this case aggregate wealth
    # 注册要计算的矩——这里是总财富
    hh = lift_moments(hh; K_supplied = at_end(integrand = :wealth, reduce = sum))

    return hh
end

"Compute interest rate r and wage w given aggregate capital K and parameters p. / 给定总资本 K 和参数 p，计算利率 r 与工资 w。"
function aiyagari_prices(K, p::AiyagariParams)
    (; α, δ, L, A) = p
    return (;
        r = A * α * (K / L)^(α - 1) - δ,
        w = A * (1 - α) * (K / L)^α
    )
end

In [5]:
hh = aiyagari_household(p)
@printf "hh: %s\n" typeof(hh).name.name

hh: ChainStage


In [6]:
aiyagari_prices(10.0, p)

(r = 0.002471235499639818, w = 1.4661552977713748)

### 2.3 Inner solve at a single $\bar K$ / 单 $\bar K$ 下的内层解

1. Construct the environment `env`: $(K, r, w)$.
2. `solve_steady_state_given_env!`: compute steady state $V$ and $\Lambda$ by value function iteration and forward simulation, respectively, given `env`.
3. `compute_moments`: Compute $K_t^{\mathrm{supplied}}$.

1. 构造环境 `env`：$(K, r, w)$。
2. `solve_steady_state_given_env!`：给定 `env`，分别用值函数迭代与前向模拟求稳态 $V$ 与 $\Lambda$。
3. `compute_moments`：计算 $K_t^{\mathrm{supplied}}$。

In [ ]:
# Allocate data buffers for the household block
# 为家庭模块分配数据缓冲区
buffers = allocate(hh)

# Guess K, construct env, solve household block, and compute K_supplied
# 猜一个 K，构造 env，求解家庭模块，并计算 K_supplied
K_guess = 5.0
env = merge((;K = K_guess), aiyagari_prices(K_guess, p))
info = solve_steady_state_given_env!(hh, env, buffers)
K_supplied_trial = compute_moments(hh, env).K_supplied

# Report results
# 输出结果
@printf "K_guess = %.4f → K_supplied = %.4f (residual = %+.4f)\n" K_guess K_supplied_trial (K_supplied_trial - K_guess)
@printf "  VFI: %d iterations; Λ: %d iterations; r = %.4f, w = %.4f\n" info.vfi_iters info.lambda_iters env.r env.w

### 2.4 Solve for steady-state $\bar K$ by iteratively guessing prices (tatonnement)
### 2.4 通过迭代猜价格求稳态 $\bar K$（tâtonnement）

In [ ]:
"Solve steady state V, Λ, *and* K given params p. / 给定参数 p，求稳态 V、Λ *以及* K。"
function aiyagari_steady_state(p::AiyagariParams; K_init=5.0, update_speed=0.01,rtol=2e-2, max_iter=500, verbosity=0)

    # Construct household and allocate buffers
    # 构造家庭模块并分配缓冲区
    hh   = aiyagari_household(p)
    buffers = allocate(hh)
    dims = layout_size(aiyagari_layout(p))

    # Guess V, Λ, and K
    # 猜测 V、Λ 与 K
    V    = zeros(Float64, dims...)
    Λ    = fill(1.0 / prod(dims), dims...)
    K = K_init

    iterations = 0
    K_err = Inf
    while iterations < max_iter
        # Solve household steady state *given* K
        # *给定* K，求家庭稳态
        env  = merge((;K), aiyagari_prices(K, p))
        (;V,Λ) = solve_steady_state_given_env!(hh, env, buffers; V_init=V, Λ_init=Λ, vfi_maxiter=10_000, vfi_tol=1e-5)

        # Compute error
        # 计算误差
        K_S = compute_moments(hh, env).K_supplied
        K_err = abs(K_S - K) / K
        verbosity >= 1 && @show K_err
        iterations += 1
        K_err <= rtol && break

        # Update K
        # 更新 K
        K += update_speed * (K_S - K)
    end
    
    return merge((; K, V, Λ, iterations), aiyagari_prices(K, p))
end

## 2.5 Comparative Statics
## 2.5 比较静态

Scenario: Permanent 5% increase in TFP.

情景：TFP 永久性提升 5%。

In [9]:
ss = aiyagari_steady_state(p; verbosity=1)
@printf "K_ss = %.4f, r_ss = %.4f, w_ss = %.4f in %d iterations\n" ss.K ss.r ss.w ss.iterations

K_err = 17.33040528646891
K_err = 0.3422047984164551
K_err = 0.3150494782885734
K_err = 0.2840128096463101
K_err = 0.27165132237330636
K_err = 0.24690771284617188
K_err = 0.2215764983736494
K_err = 0.19700979313355893
K_err = 0.18090567191478363
K_err = 0.15844662090970715
K_err = 0.13640435073701476
K_err = 0.12579388903786556
K_err = 0.10092199755258792
K_err = 0.08584132690241326
K_err = 0.07462145401379783
K_err = 0.07055134892447225
K_err = 0.06400119462721605
K_err = 0.0549031804355612
K_err = 0.050653394145975354
K_err = 0.03078648998639764
K_err = 0.027685075320643654
K_err = 0.02174691779041854
K_err = 0.014684485002136346
K_ss = 5.6894, r_ss = 0.0383, w_ss = 1.1968 in 23 iterations


In [10]:
new_ss = aiyagari_steady_state((@set p.A = 1.05); K_init=6.0, verbosity=1)
@printf "K_ss = %.4f, r_ss = %.4f, w_ss = %.4f in %d iterations\n" new_ss.K new_ss.r new_ss.w new_ss.iterations

K_err = 1.7204627710116522
K_err = 0.2263040006787007
K_err = 0.11493740952722771
K_err = 0.06963336415024239
K_err = 0.03395722999535803
K_err = 0.02471173897242368
K_err = 0.015047548789380657
K_ss = 6.1319, r_ss = 0.0384, w_ss = 1.2909 in 7 iterations


## 3. MIT Shock
## 3. MIT 冲击

### 3.1 Example — TFP shock / 我们的示例：TFP 冲击

A permanent positive 5% increase in TFP.

TFP 永久性正向提升 5%。

## 4. Solving an MIT transition
## 4. 解 MIT 过渡

**Given.** The exogenous path $\{A_t\}_{t=1}^T$, the household chain `hh`, and the firm $(\alpha, \delta, L)$.

**Find.** Sequences $\{K_t\}, \{V_t\}, \{\Lambda_t\}$ such that
- $V_{T+1} = V_{\mathrm{ss\_new}}$ (terminal condition),
- $\Lambda_1 = \Lambda_{\mathrm{ss\_old}}$ (initial condition),
- $V_t = \texttt{backward!}(V_{t+1}, \mathrm{env}_t)$,
- $\Lambda_{t+1} = \texttt{forward!}(\Lambda_t)$,
- $K_t = \int b\,\mathrm{d}\Lambda_t = K_t^{\mathrm{supplied}}$ (market clears every period).

**Algorithm.**

1. **Guess $\{K_t\}$.**
1. **Iterate $V$ backward.** $V_T \leftarrow V_{\mathrm{ss}}$, then iterate backward.
2. **Iterate $\Lambda$ forward.** $\Lambda_1 \leftarrow \Lambda_{\mathrm{ss}}$, then iterate forward.
3. **Outer tatonnement on $\{K_t\}$.** Update: $K_t \leftarrow (1-d) K_t + d K_t^{\mathrm{supplied}}$.

**已知。** 外生路径 $\{A_t\}_{t=1}^T$、家庭链 `hh`、企业 $(\alpha, \delta, L)$。

**求。** 序列 $\{K_t\}, \{V_t\}, \{\Lambda_t\}$ 满足
- $V_{T+1} = V_{\mathrm{ss\_new}}$（末端条件），
- $\Lambda_1 = \Lambda_{\mathrm{ss\_old}}$（初始条件），
- $V_t = \texttt{backward!}(V_{t+1}, \mathrm{env}_t)$，
- $\Lambda_{t+1} = \texttt{forward!}(\Lambda_t)$，
- $K_t = \int b\,\mathrm{d}\Lambda_t = K_t^{\mathrm{supplied}}$（每期市场出清）。

**算法。**

1. **猜 $\{K_t\}$。**
1. **后向迭代 $V$。** $V_T \leftarrow V_{\mathrm{ss}}$，然后向后迭代。
2. **前向迭代 $\Lambda$。** $\Lambda_1 \leftarrow \Lambda_{\mathrm{ss}}$，然后向前迭代。
3. **对 $\{K_t\}$ 做外层 tâtonnement。** 更新：$K_t \leftarrow (1-d) K_t + d K_t^{\mathrm{supplied}}$。

In [ ]:
function mit_shock_transition(p::AiyagariParams;
                               T::Int       = 100,
                               A_new::Float64 = 1.05,
                               update_speed = 0.2,
                               tol          = 1e-3,
                               max_iter     = 200,
                               verbosity    = 0)
    # Solve initial and final steady states
    # 求初始和最终稳态
    (;K, V, Λ) = aiyagari_steady_state(p)
    params_new = @set p.A = A_new
    ss_new = aiyagari_steady_state(params_new; K_init=K*A_new)

    # Construct household block
    # 构造家庭模块
    hh = aiyagari_household(params_new)
    dims = layout_size(aiyagari_layout(params_new))
    
    # Allocate data paths
    # 分配数据路径
    buffers = [allocate(hh) for _ in 1:T]
    V_path = [copy(V) for _ in 1:T]
    Λ_path = [zeros(Float64, dims...) for _ in 1:T]
    K_guess_path = fill(K, T)
    K_S_path = zeros(T)

    # Set boundary conditions
    # 设定边界条件
    V_path[T] .= ss_new.V
    Λ_path[1] .= Λ

    # Exogenous TFP path; initial K-path guess = constant SS.
    # 外生 TFP 路径；初始 K 路径猜测 = 常稳态。
    iterations = 0
    K_path_err = Inf

    while iterations < max_iter
        env_path = [merge((;K=K_guess_path[t]), aiyagari_prices(K_guess_path[t], params_new)) for t=1:T]
        # Iterate V backward: V_ss_new = V_T → V_{T-1} → … → V_1.
        # V 向后迭代：V_ss_new = V_T → V_{T-1} → … → V_1。
        for t in reverse(1:T-1)
            V_path[t] .= backward!(hh, V_path[t + 1], env_path[t], buffers[t])
        end

        # Iterate Λ forward: Λ_ss = Λ_1 → Λ_2 → … → Λ_T.
        # Λ 向前迭代：Λ_ss = Λ_1 → Λ_2 → … → Λ_T。
        for t in 1:T-1
            Λ_path[t+1] .= forward!(hh, Λ_path[t], buffers[t])
            K_S_path[t] = compute_moments(hh, env_path[t]).K_supplied
        end

        # Check error
        # 检查误差
        K_path_err = maximum(abs, K_S_path .- K_guess_path)
        iterations += 1
        K_path_err <= tol && break
        verbosity >= 1 && @show K_path_err

        # Update K_path
        # 更新 K_path
        K_guess_path .= update_speed .* K_guess_path .+ (1 - update_speed) .* K_S_path
    end

    @assert K_path_err <= tol
    return (; K_path = K_guess_path, V_path, Λ_path, K, V, Λ, iterations)
end

In [ ]:
mit_shock_transition(p; A_new=1.05, verbosity=1)

### 5.5 Run it / 跑一下

In [ ]:
tr = mit_shock_transition(p; T = T, A_0 = 1.05, ρ = 0.85, damping = 0.2, tol = 1e-3, max_iter = 200)
@printf "converged = %s in %d outer iterations\n" tr.converged tr.iterations
@printf "K_ss            = %.4f\n" tr.K_ss
@printf "K[1]   (impact) = %.4f  (Δ = %+0.4f)\n" tr.K_path[1] (tr.K_path[1] - tr.K_ss)
@printf "K[5]            = %.4f  (Δ = %+0.4f)\n" tr.K_path[5] (tr.K_path[5] - tr.K_ss)
@printf "K[20]           = %.4f  (Δ = %+0.4f)\n" tr.K_path[20] (tr.K_path[20] - tr.K_ss)
@printf "K[100] (≈end)   = %.4f  (Δ = %+0.4f)\n" tr.K_path[100] (tr.K_path[100] - tr.K_ss)

### 5.6 Impulse responses / 脉冲响应

Capital first; then $r$ and $w$.

先看资本，再看 $r$ 与 $w$。

In [ ]:
plot(1:T, tr.K_path; lw = 2, label = "K_t (transition)",
     xlabel = "period t", ylabel = "aggregate capital",
     title = "IRF: K to a +5% TFP shock with ρ = 0.85")
hline!([tr.K_ss]; color = :gray, linestyle = :dash, label = "K_ss")

In [ ]:
r_path = [mit_prices(tr.K_path[t], tr.A_path[t], p).r for t in 1:T]
w_path = [mit_prices(tr.K_path[t], tr.A_path[t], p).w for t in 1:T]

plot(layout = (2, 1), size = (700, 500))
plot!(1:T, r_path; subplot = 1, lw = 2, label = "r_t",
      xlabel = "period", ylabel = "r", title = "IRF: real rate")
hline!([mit_prices(tr.K_ss, 1.0, p).r];
       subplot = 1, color = :gray, linestyle = :dash, label = "r_ss")
plot!(1:T, w_path; subplot = 2, lw = 2, label = "w_t",
      xlabel = "period", ylabel = "w", title = "IRF: wage")
hline!([mit_prices(tr.K_ss, 1.0, p).w];
       subplot = 2, color = :gray, linestyle = :dash, label = "w_ss")

### 5.7 Residual history / 残差历史

Damped tatonnement drops the residual geometrically until it hits a *discretization floor* at
$\sim 2.5 \times 10^{-3}$. The floor is the hard-`argmax` `ConsumptionSavingsStage` policy flipping
between adjacent grid cells as $K_t$ wobbles. A smoothed (`LogitChoiceStage`-based) savings policy
or a tighter wealth grid would push the floor down.

阻尼 tatonnement 残差几何下降，直到撞上 $\sim 2.5 \times 10^{-3}$ 的*离散化下限*。
下限来自硬 `argmax` 的 `ConsumptionSavingsStage`：$K_t$ 微动时策略在相邻格子间翻转。
把储蓄换成平滑（`LogitChoiceStage`）或加密财富格点，下限就会下降。

In [ ]:
plot(1:length(tr.residual_history), tr.residual_history;
     yscale = :log10, lw = 2, marker = :circle, markersize = 3,
     xlabel = "outer iteration", ylabel = "‖K^supplied − K‖∞",
     label = "residual", title = "Damped tatonnement residual history")

### 5.8 Try this — damping sweep / 动手试 —— 阻尼扫一遍

Damping is a craft: too high oscillates, too low crawls. Re-run the transition at
$d \in \{0.1, 0.2, 0.4, 0.6\}$ and overlay the residual histories.

阻尼是个手艺活：太大振荡，太小爬不动。把过渡在 $d \in \{0.1, 0.2, 0.4, 0.6\}$
重新跑几遍，把残差历史叠在一张图上看看。

**Expected:** $d \in \{0.4, 0.6\}$ fails to converge (residual oscillates or grows);
$d = 0.1$ converges with a shallow slope; $d = 0.2$ is roughly the sweet spot.

**预期：**$d \in \{0.4, 0.6\}$ 不收敛（残差振荡或增大）；$d = 0.1$ 缓慢线性下降；
$d = 0.2$ 大约是最佳位置。

In [ ]:
plt = plot(yscale = :log10, xlabel = "outer iteration",
           ylabel = "residual ‖K^supplied − K‖∞",
           title  = "Damping sweep")
for d in (0.1, 0.2, 0.4, 0.6)
    res = mit_shock_transition(p; T = T, A_0 = 1.05, ρ = 0.85,
                                 damping = d, tol = 1e-3, max_iter = 80)
    plot!(plt, 1:length(res.residual_history), res.residual_history;
          lw = 2, label = "d = $(d)")
end
plt